In [1]:
!pip install -U spacy
!python -m spacy download ru_core_news_sm
!pip install faiss-cpu
!pip install -U rank-bm25 pymorphy2 pymorphy2-dict-ru
!pip install -U sentence-transformers transformers accelerate bitsandbytes

  Using cached https://github.com/explosion/spacy-models/releases/download/ru_core_news_sm-3.8.0/ru_core_news_sm-3.8.0-py3-none-any.whl (15.3 MB)
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
ERROR: Could not find a version that satisfies the requirement pymorphy2-dict-ru (from versions: none)
ERROR: No matching distribution found for pymorphy2-dict-ru


In [2]:
!pip install -U sentence-transformers

In [3]:
!pip install -U bitsandbytes

In [4]:
# === CELL 1 — Google Drive / paths ===
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped or failed:', e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# === CELL 2 — imports / seed / device ===
import os
import re
import gc
import html
import json
import time
import math
import pickle
import hashlib
import inspect
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple, Iterable
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

warnings.filterwarnings('ignore')

# pymorphy2 compatibility for Python 3.11/3.12
if not hasattr(inspect, 'getargspec'):
    from collections import namedtuple
    ArgSpec = namedtuple('ArgSpec', 'args varargs keywords defaults')
    def getargspec(func):
        spec = inspect.getfullargspec(func)
        return ArgSpec(args=spec.args, varargs=spec.varargs, keywords=spec.varkw, defaults=spec.defaults)
    inspect.getargspec = getargspec

try:
    import pymorphy2
    MORPH = pymorphy2.MorphAnalyzer()
except Exception as e:
    MORPH = None
    print('pymorphy2 unavailable, lemmatization disabled:', e)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

DEVICE: cuda
GPU: NVIDIA L4


In [6]:
# === CELL 3 — global configuration ===
DATA_DIR = Path('/content/drive/MyDrive/RAG')
QUESTIONS_PATH = DATA_DIR / 'questions.csv'
WEBSITES_PATH = DATA_DIR / 'websites.csv'
SAMPLE_SUBMISSION_PATH = DATA_DIR / 'sample_submission.csv'

RUN_NAME = 'qwen_generator_first_v1'
SAVE_DIR = Path('/content/alfa_rag_qwen_cache')
CACHE_DIR = SAVE_DIR / RUN_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SUBMISSION_PATH = DATA_DIR / f'submission_{RUN_NAME}.csv'
DIAGNOSTICS_PATH = DATA_DIR / f'diagnostics_{RUN_NAME}.csv'
CHECKPOINT_PATH = CACHE_DIR / 'answers_checkpoint.csv'

# Быстрый режим для отладки. Поставьте None для полного сабмита.
DEV_N: Optional[int] = None       # например 100 для теста
FORCE_REBUILD_INDEX = False
FORCE_REBUILD_CHUNKS = False
FORCE_REGENERATE = False

# Qwen-first stack. Для L4/T4 начните с 0.6B embedding + 0.6B reranker + 7B generator.
# Если есть A100 и время, попробуйте Qwen/Qwen3-Embedding-4B и Qwen/Qwen3-8B.
EMBEDDING_MODEL_NAME = 'Qwen/Qwen3-Embedding-0.6B'
RERANKER_MODEL_NAME = 'Qwen/Qwen3-Reranker-0.6B'
GENERATOR_MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
# Альтернатива: GENERATOR_MODEL_NAME = 'Qwen/Qwen3-8B'

NO_ANSWER = 'Нет ответа.'

CHUNKING_CONFIG = {
    'min_chunk_chars': 140,
    'target_chunk_chars': 760,
    'max_chunk_chars': 1050,
    'overlap_chars': 120,
    'max_line_chars_for_heading': 110,
    'min_text_chars': 40,
    'max_special_ratio': 0.48,
    'max_digit_ratio': 0.78,
}

RETRIEVAL_CONFIG = {
    'semantic_top_n': 120,
    'bm25_top_n': 120,
    'final_retrieval_top_n': 120,
    'rrf_k': 60,
    'use_lemmatization': True,
    'embedding_batch_size': 16 if DEVICE == 'cuda' else 4,
    'query_batch_size': 16,
}

RERANKER_CONFIG = {
    'input_top_n': 100,
    'rerank_top_k': 48,
    'batch_size': 4 if DEVICE == 'cuda' else 1,
    'max_length': 4096,
    'use_4bit': False,  # 0.6B обычно помещается без 4bit; для 4B можно поставить True.
}

EVIDENCE_CONFIG = {
    'final_top_k': 8,
    'max_chunks_per_web_id': 2,
    'neighbor_window': 1,
    'max_expanded_chunks': 16,
    'max_context_chars_default': 3800,
    'max_context_chars_legal': 5000,
    'max_context_chars_tariff': 5000,
    'max_context_chars_procedure': 4300,
    'max_evidence_sentences': 24,
}

GENERATION_CONFIG = {
    'load_in_4bit': True,
    'max_new_tokens': 240,
    'temperature': 0.15,
    'top_p': 0.85,
    'repetition_penalty': 1.03,
    'do_sample': True,
    'answer_cap_default': 850,
    'answer_cap_fact': 760,
    'answer_cap_procedure': 900,
    'answer_cap_legal': 1050,
    'answer_cap_tariff': 1050,
    'save_every': 25,
}

ANSWERABILITY_CONFIG = {
    # Пороги намеренно мягкие, чтобы не плодить false no-answer.
    'min_top_sigmoid': 0.18,
    'min_query_overlap': 0.035,
    'min_context_chars': 120,
    'very_low_top_sigmoid': 0.08,
}

print('CACHE_DIR:', CACHE_DIR)
print('SUBMISSION_PATH:', SUBMISSION_PATH)

CACHE_DIR: /content/alfa_rag_qwen_cache/qwen_generator_first_v1
SUBMISSION_PATH: /content/drive/MyDrive/RAG/submission_qwen_generator_first_v1.csv


In [7]:
# === CELL 4 — data loading ===
def read_csv_safely(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')
    return pd.read_csv(path)

questions = read_csv_safely(QUESTIONS_PATH)
websites = read_csv_safely(WEBSITES_PATH)
sample_submission = read_csv_safely(SAMPLE_SUBMISSION_PATH) if SAMPLE_SUBMISSION_PATH.exists() else None

# Поддержка обеих схем: text/web, url/website.
if 'text' not in websites.columns and 'web' in websites.columns:
    websites = websites.rename(columns={'web': 'text'})
if 'url' not in websites.columns and 'website' in websites.columns:
    websites = websites.rename(columns={'website': 'url'})

required_q = {'q_id', 'query'}
required_w = {'web_id', 'url', 'title', 'text'}
assert required_q.issubset(questions.columns), f'questions columns: {questions.columns}'
assert required_w.issubset(websites.columns), f'websites columns: {websites.columns}'

questions['q_id'] = questions['q_id'].astype(int)
questions['query'] = questions['query'].fillna('').astype(str)
websites['web_id'] = websites['web_id'].astype(int)
websites['title'] = websites['title'].fillna('').astype(str)
websites['text'] = websites['text'].fillna('').astype(str)
websites['url'] = websites['url'].fillna('').astype(str)
if 'kind' not in websites.columns:
    websites['kind'] = ''
websites['kind'] = websites['kind'].fillna('').astype(str)

if DEV_N is not None:
    questions_run = questions.head(int(DEV_N)).copy()
else:
    questions_run = questions.copy()

print('questions:', questions.shape, 'run:', questions_run.shape)
print('websites:', websites.shape)
print('sample_submission:', None if sample_submission is None else sample_submission.shape)
display(questions.head())
display(websites.head())

questions: (6977, 2) run: (6977, 2)
websites: (1937, 5)
sample_submission: (6977, 2)


,q_id,query
0,1,Номер счета
1,2,Где узнать бик и счёт
2,3,Мне не приходят коды для подтверждения данной ...
3,4,"Оформила рассрочку ,но уведомлений никаких не ..."
4,5,"Здравствуйте, когда смогу пользоваться кредитн..."


,web_id,url,kind,title,text
0,1,https://alfabank.ru/,html,"Альфа-Банк - кредитные и дебетовые карты, кред...",Рассчитайте выгоду\nРасчёт калькулятора предва...
1,2,https://alfabank.ru/a-club/,html,А-Клуб. Деньги имеют значение,Брокерские услуги\nОткрытие брокерского счёта ...
2,3,https://alfabank.ru/a-club/ultimate/,html,А-Клуб. Деньги имеют значение,Хотите получить больше информации?\nПозвоните ...
3,4,https://alfabank.ru/actions/rules/,html,Скидки по картам,Правила проведения Акции «Альфа Пятница. Бараб...
4,5,https://alfabank.ru/alfafuture/,html,Альфа‑Будущее: Платформа для развития студенто...,Образование\nМагистратуры\nМагистратура ВШЭ\nМ...


In [8]:
# === CELL 5 — preprocessing utilities ===
RUS_STOP = set('''и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между'''.split())

TERM_NORMALIZATION = {
    'ё': 'е',
    'смс': 'sms',
    'с м с': 'sms',
    'пуш': 'push',
    'пуши': 'push',
    'push-уведомления': 'push уведомления',
    'пин': 'pin',
    'пин-код': 'pin-код',
    'альфа онлайн': 'альфа-онлайн',
    'альфа банк': 'альфа-банк',
    'альфабанк': 'альфа-банк',
}

BOILERPLATE_PATTERNS = [
    r'^\s*cookie', r'используем cookie', r'файлы cookie', r'принять все',
    r'^\s*меню\s*$', r'^\s*главная\s*$', r'^\s*назад\s*$', r'^\s*вверх\s*$',
    r'скачайте приложение', r'установите приложение', r'отсканируйте qr',
    r'поделиться$', r'читать также', r'смотрите также', r'похожие статьи',
    r'©', r'альфа-банк использует', r'продолжая пользоваться сайтом',
    r'лицензия банка россии', r'генеральная лицензия',
]
BOILERPLATE_RE = re.compile('|'.join(f'(?:{p})' for p in BOILERPLATE_PATTERNS), re.IGNORECASE)

LEGAL_HINT_RE = re.compile(r'\b(договор|условия|тариф|тарифы|комисси|ставк|пени|неустойк|пункт|раздел|приложени[ея]|правил[ао]|заявлени[ея])\b', re.I)
FAQ_HINT_RE = re.compile(r'\?|\b(как|где|почему|что делать|можно ли|сколько|когда|какой|какая|какие)\b', re.I)
SUPPORT_HINT_RE = re.compile(r'\b(помощь|поддержк|вопрос|ответ|инструкц|как сделать|не работает|ошибк|заблок|восстанов|сменить|отключить|подключить)\b', re.I)
TARIFF_HINT_RE = re.compile(r'\b(тариф|комисс|стоимост|бесплатн|платн|лимит|процент|ставк|руб|₽|%|кэшб[эе]к|cashback)\b', re.I)


def normalize_terms(text: str) -> str:
    t = str(text)
    for src, dst in TERM_NORMALIZATION.items():
        t = re.sub(re.escape(src), dst, t, flags=re.IGNORECASE)
    return t


def basic_clean_text(text: str) -> str:
    t = html.unescape(str(text))
    t = t.replace('\u00a0', ' ').replace('\u200b', '')
    t = t.replace('‑', '-').replace('–', '-').replace('—', ' - ')
    t = re.sub(r'<[^>]+>', ' ', t)
    t = re.sub(r'(?<=\w)-\s*\n\s*(?=\w)', '', t)
    t = re.sub(r'\r', '\n', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    t = normalize_terms(t)
    # Восстановление пробелов между числами/словами и после пунктуации.
    t = re.sub(r'(?<=[а-яА-Яa-zA-Z])(?=\d)', ' ', t)
    t = re.sub(r'(?<=\d)(?=[а-яА-Яa-zA-Z])', ' ', t)
    t = re.sub(r'(?<=[,;:])(?=\S)', ' ', t)
    t = re.sub(r'\s+', ' ', t.replace('\n', ' \n '))
    t = re.sub(r'\s*\n\s*', '\n', t)
    return t.strip()


def dedupe_lines(text: str) -> str:
    lines = [re.sub(r'\s+', ' ', ln).strip() for ln in str(text).split('\n')]
    out, seen = [], set()
    for ln in lines:
        if not ln:
            if out and out[-1] != '':
                out.append('')
            continue
        low = ln.lower().strip()
        if BOILERPLATE_RE.search(low):
            continue
        # Очень короткие навигационные строки часто шумят.
        if len(low) <= 2:
            continue
        key = re.sub(r'\W+', '', low)
        if key in seen and len(low) < 160:
            continue
        seen.add(key)
        out.append(ln)
    return '\n'.join(out).strip()


def text_quality_ok(text: str) -> bool:
    t = str(text).strip()
    if len(t) < CHUNKING_CONFIG['min_text_chars']:
        return False
    special = sum(1 for c in t if not (c.isalnum() or c.isspace() or c in '.,;:!?%₽/\\-()[]+№"«»'))
    digits = sum(1 for c in t if c.isdigit())
    if special / max(len(t), 1) > CHUNKING_CONFIG['max_special_ratio']:
        return False
    if digits / max(len(t), 1) > CHUNKING_CONFIG['max_digit_ratio']:
        return False
    return True


def source_type(url: str, title: str, text: str) -> str:
    u = str(url).lower()
    tt = (str(title) + ' ' + str(text[:1000])).lower()
    if LEGAL_HINT_RE.search(tt) or any(x in u for x in ['/tariffs', '/legal', '/documents', '/dogovor', '/pravila']):
        if TARIFF_HINT_RE.search(tt):
            return 'legal_tariff'
        return 'legal'
    if any(x in u for x in ['/help', '/support', '/faq', '/questions']) or SUPPORT_HINT_RE.search(tt):
        return 'support_faq' if FAQ_HINT_RE.search(tt) else 'support_help'
    if any(x in u for x in ['/news', '/press']) or 'новост' in tt:
        return 'news_article'
    if any(x in u for x in ['/blog', '/journal']) or 'блог' in tt:
        return 'blog_article'
    if any(x in u for x in ['/cards', '/credit', '/debit', '/ipoteka', '/loans', '/invest', '/business', '/sme']):
        return 'product_page'
    return 'general_page'


def url_path_tokens(url: str) -> str:
    u = re.sub(r'https?://[^/]+', '', str(url).lower())
    u = re.sub(r'[^a-zа-я0-9]+', ' ', u)
    return u.strip()


def stable_hash(text: str) -> str:
    return hashlib.md5(str(text).encode('utf-8')).hexdigest()

In [9]:
# === CELL 6 — section-aware chunking ===
@dataclass
class Chunk:
    chunk_id: int
    web_id: int
    url: str
    title: str
    source_type: str
    section_title: str
    chunk_index: int
    text: str
    retrieval_text: str
    text_hash: str


def split_sentences(text: str) -> List[str]:
    t = re.sub(r'\s+', ' ', str(text)).strip()
    if not t:
        return []
    # Не идеально, но достаточно устойчиво для русских FAQ/страниц.
    parts = re.split(r'(?<=[.!?])\s+(?=[А-ЯA-ZЁ0-9])', t)
    out = []
    for p in parts:
        p = p.strip()
        if len(p) > 0:
            out.append(p)
    return out


def is_heading(line: str) -> bool:
    s = line.strip()
    if not s:
        return False
    if len(s) > CHUNKING_CONFIG['max_line_chars_for_heading']:
        return False
    if s.endswith(('.', ',', ';', ':')) and len(s) > 25:
        return False
    alpha = sum(ch.isalpha() for ch in s)
    if alpha < 3:
        return False
    # Заголовки часто без точки, короткие, с заглавной буквы или FAQ-вопросы.
    return s.endswith('?') or not re.search(r'[.!]$', s)


def build_sections(title: str, text: str) -> List[Tuple[str, str]]:
    lines = [ln.strip() for ln in str(text).split('\n')]
    sections = []
    current_title = title.strip() or 'Документ'
    buf = []
    for ln in lines:
        if not ln:
            continue
        if is_heading(ln) and len(buf) >= 1:
            body = ' '.join(buf).strip()
            if body:
                sections.append((current_title, body))
            current_title = ln
            buf = []
        else:
            buf.append(ln)
    body = ' '.join(buf).strip()
    if body:
        sections.append((current_title, body))
    if not sections and text.strip():
        sections = [(title.strip() or 'Документ', text.strip())]
    return sections


def make_faq_projection(title: str, section_title: str, text: str, stype: str) -> str:
    first = ' '.join(split_sentences(text)[:3])[:500]
    bits = []
    if title:
        bits.append(f'Тема страницы: {title}.')
    if section_title and section_title != title:
        bits.append(f'Раздел: {section_title}.')
    if stype:
        bits.append(f'Тип источника: {stype}.')
    if FAQ_HINT_RE.search(title + ' ' + section_title + ' ' + first):
        bits.append(f'Вопрос пользователя может быть про: {section_title or title}. Ответ находится в тексте: {first}')
    else:
        bits.append(f'Краткое содержание: {first}')
    return ' '.join(bits).strip()


def make_retrieval_text(url: str, title: str, section_title: str, text: str, stype: str) -> str:
    projection = make_faq_projection(title, section_title, text, stype)
    path = url_path_tokens(url)
    # Для dense retrieval добавляем поля, но финальный ответ потом строится только по original text.
    return '\n'.join([
        f'url_path: {path}',
        f'source_type: {stype}',
        f'title: {title}',
        f'section: {section_title}',
        projection,
        f'body: {text}',
    ]).strip()


def chunk_section(text: str, target: int, max_chars: int, overlap_chars: int) -> List[str]:
    sentences = split_sentences(text)
    if not sentences:
        return []
    chunks = []
    cur = ''
    for sent in sentences:
        if len(cur) + len(sent) + 1 <= max_chars:
            cur = (cur + ' ' + sent).strip()
        else:
            if len(cur) >= CHUNKING_CONFIG['min_chunk_chars']:
                chunks.append(cur.strip())
            # overlap по хвосту предыдущего чанка
            tail = cur[-overlap_chars:] if overlap_chars and cur else ''
            cur = (tail + ' ' + sent).strip()
            if len(cur) > max_chars:
                cur = cur[-max_chars:]
        if len(cur) >= target and re.search(r'[.!?]$', cur.strip()):
            chunks.append(cur.strip())
            cur = cur[-overlap_chars:].strip() if overlap_chars else ''
    if len(cur.strip()) >= CHUNKING_CONFIG['min_chunk_chars']:
        chunks.append(cur.strip())
    # cleanup tiny duplicates
    cleaned = []
    seen = set()
    for c in chunks:
        c = re.sub(r'\s+', ' ', c).strip()
        key = stable_hash(c.lower())
        if key not in seen and text_quality_ok(c):
            cleaned.append(c)
            seen.add(key)
    return cleaned


def build_chunks(websites_df: pd.DataFrame) -> List[Chunk]:
    chunks: List[Chunk] = []
    seen_hashes = set()
    chunk_id = 0
    for _, row in tqdm(websites_df.iterrows(), total=len(websites_df), desc='Building chunks'):
        web_id = int(row['web_id'])
        url = str(row['url'])
        title = basic_clean_text(row['title'])
        raw_text = basic_clean_text(row['text'])
        text = dedupe_lines(raw_text)
        if not text_quality_ok(text):
            continue
        stype = source_type(url, title, text)
        sections = build_sections(title, text)
        local_idx = 0
        for section_title, section_body in sections:
            section_title = basic_clean_text(section_title)[:180]
            for ctext in chunk_section(section_body, CHUNKING_CONFIG['target_chunk_chars'], CHUNKING_CONFIG['max_chunk_chars'], CHUNKING_CONFIG['overlap_chars']):
                norm_hash = stable_hash(re.sub(r'\W+', '', ctext.lower())[:4000])
                # exact/near-exact dedupe, но не удаляем одинаковые короткие условия из разных источников слишком агрессивно
                if norm_hash in seen_hashes and len(ctext) < 700:
                    continue
                seen_hashes.add(norm_hash)
                rtext = make_retrieval_text(url, title, section_title, ctext, stype)
                chunks.append(Chunk(
                    chunk_id=chunk_id,
                    web_id=web_id,
                    url=url,
                    title=title,
                    source_type=stype,
                    section_title=section_title,
                    chunk_index=local_idx,
                    text=ctext,
                    retrieval_text=rtext,
                    text_hash=norm_hash,
                ))
                chunk_id += 1
                local_idx += 1
    return chunks

chunks_path = CACHE_DIR / 'chunks.pkl'
if chunks_path.exists() and not FORCE_REBUILD_CHUNKS:
    with open(chunks_path, 'rb') as f:
        chunks = pickle.load(f)
    print('Loaded chunks:', len(chunks))
else:
    chunks = build_chunks(websites)
    with open(chunks_path, 'wb') as f:
        pickle.dump(chunks, f)
    print('Saved chunks:', len(chunks))

chunk_df = pd.DataFrame([asdict(c) for c in chunks])
print(chunk_df['source_type'].value_counts())
display(chunk_df.head())

Building chunks:   0%|          | 0/1937 [00:00<?, ?it/s]

Saved chunks: 11095
source_type
legal           4393
support_faq     2742
legal_tariff    1760
product_page    1233
general_page     637
support_help     255
blog_article      53
news_article      22
Name: count, dtype: int64


,chunk_id,web_id,url,title,source_type,section_title,chunk_index,text,retrieval_text,text_hash
0,0,1,https://alfabank.ru/,"Альфа-Банк - кредитные и дебетовые карты, кред...",legal,"Альфа-Банк - кредитные и дебетовые карты, кред...",0,Рассчитайте выгоду Расчет калькулятора предвар...,url_path: \nsource_type: legal\ntitle: Альфа-Б...,770fe43244b1399edf29c1f5f204d884
1,1,4,https://alfabank.ru/actions/rules/,Скидки по картам,legal_tariff,Скидки по картам,0,Правила проведения Акции «Альфа Пятница. Бараб...,url_path: actions rules\nsource_type: legal_ta...,3da264155bc768f9013a9ac90ab6cd2f
2,2,5,https://alfabank.ru/alfafuture/,Альфа-Будущее: Платформа для развития студенто...,general_page,Альфа-Будущее: Платформа для развития студенто...,0,Образование Магистратуры Магистратура ВШЭ Маги...,url_path: alfafuture\nsource_type: general_pag...,dbe087f1da548159219d49d17f26aabe
3,3,6,https://alfabank.ru/alfafuture/blog/alfabank-l...,Дневник,blog_article,Дневник,0,Альфа-Банк занял 1 место в одном из главных мо...,url_path: alfafuture blog alfabank luchshiy ba...,f714ee874a80526ac500575aba1762eb
4,4,6,https://alfabank.ru/alfafuture/blog/alfabank-l...,Дневник,blog_article,Дневник,1,Company Award - это ежегодная премия в сфере р...,url_path: alfafuture blog alfabank luchshiy ba...,8d382a4ea9a02fefea9642b30266dac7


In [10]:
# === CELL 7 — tokenization / BM25 ===
TOKEN_RE = re.compile(r'[a-zA-Zа-яА-ЯёЁ0-9%₽]+')
LEMMA_CACHE: Dict[str, str] = {}


def tokenize(text: str, lemmatize: bool = True) -> List[str]:
    text = normalize_terms(str(text).lower())
    toks = TOKEN_RE.findall(text)
    out = []
    for tok in toks:
        if len(tok) <= 1 or tok in RUS_STOP:
            continue
        if lemmatize and MORPH is not None and re.search('[а-яё]', tok):
            if tok not in LEMMA_CACHE:
                try:
                    LEMMA_CACHE[tok] = MORPH.parse(tok)[0].normal_form
                except Exception:
                    LEMMA_CACHE[tok] = tok
            tok = LEMMA_CACHE[tok]
        out.append(tok)
    return out

bm25_path = CACHE_DIR / 'bm25.pkl'
tokens_path = CACHE_DIR / 'bm25_tokens.pkl'
if bm25_path.exists() and tokens_path.exists() and not FORCE_REBUILD_INDEX:
    with open(bm25_path, 'rb') as f:
        bm25 = pickle.load(f)
    with open(tokens_path, 'rb') as f:
        bm25_tokens = pickle.load(f)
    print('Loaded BM25 tokens:', len(bm25_tokens))
else:
    bm25_tokens = [tokenize(c.retrieval_text, RETRIEVAL_CONFIG['use_lemmatization']) for c in tqdm(chunks, desc='Tokenizing chunks')]
    bm25 = BM25Okapi(bm25_tokens)
    with open(bm25_path, 'wb') as f:
        pickle.dump(bm25, f)
    with open(tokens_path, 'wb') as f:
        pickle.dump(bm25_tokens, f)
    print('Saved BM25:', len(bm25_tokens))

Tokenizing chunks:   0%|          | 0/11095 [00:00<?, ?it/s]

Saved BM25: 11095


In [11]:
# === CELL 8 — Qwen embedding model / FAISS index ===
class QwenEmbedder:
    def __init__(self, model_name: str, device: str = DEVICE):
        self.model_name = model_name
        self.device = device
        print('Loading embedding model:', model_name)
        self.model = SentenceTransformer(model_name, device=device, trust_remote_code=True)
        self.query_prompt = 'Instruct: Given a question from an Alfa-Bank user, retrieve relevant Russian passages that answer it.\nQuery: '

    def encode(self, texts: List[str], is_query: bool = False, batch_size: int = 16) -> np.ndarray:
        if not texts:
            return np.zeros((0, self.get_dim()), dtype='float32')
        kwargs = dict(
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        # Qwen3/GTE instruct models benefit from explicit query prompt.
        if is_query:
            try:
                emb = self.model.encode(texts, prompt=self.query_prompt, **kwargs)
            except TypeError:
                emb = self.model.encode([self.query_prompt + t for t in texts], **kwargs)
        else:
            emb = self.model.encode(texts, **kwargs)
        return np.asarray(emb, dtype='float32')

    def get_dim(self) -> int:
        return self.model.get_sentence_embedding_dimension()

embedder = QwenEmbedder(EMBEDDING_MODEL_NAME)

emb_path = CACHE_DIR / f'embeddings_{EMBEDDING_MODEL_NAME.replace("/", "__")}.npy'
faiss_path = CACHE_DIR / f'faiss_{EMBEDDING_MODEL_NAME.replace("/", "__")}.index'

if emb_path.exists() and faiss_path.exists() and not FORCE_REBUILD_INDEX:
    doc_embeddings = np.load(emb_path)
    index = faiss.read_index(str(faiss_path))
    print('Loaded embeddings/index:', doc_embeddings.shape)
else:
    retrieval_texts = [c.retrieval_text for c in chunks]
    all_emb = []
    bs = RETRIEVAL_CONFIG['embedding_batch_size']
    for i in tqdm(range(0, len(retrieval_texts), bs), desc='Encoding chunks'):
        all_emb.append(embedder.encode(retrieval_texts[i:i+bs], is_query=False, batch_size=bs))
    doc_embeddings = np.vstack(all_emb).astype('float32')
    index = faiss.IndexFlatIP(doc_embeddings.shape[1])
    index.add(doc_embeddings)
    np.save(emb_path, doc_embeddings)
    faiss.write_index(index, str(faiss_path))
    print('Built and saved embeddings/index:', doc_embeddings.shape)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading embedding model: Qwen/Qwen3-Embedding-0.6B


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Encoding chunks:   0%|          | 0/694 [00:00<?, ?it/s]

Built and saved embeddings/index: (11095, 1024)


In [12]:
# === CELL 9 — query normalization / intent / variants ===
def normalize_query(q: str) -> str:
    q = basic_clean_text(q)
    q = re.sub(r'\s+', ' ', q).strip()
    return q


def classify_intent(q: str) -> str:
    ql = normalize_terms(q.lower())
    if re.search(r'\b(я|мне|мой|моя|мои|меня|у меня|могу ли я|одобрили|статус|заявк|доступн[а-я]* мне|почему мне)\b', ql):
        return 'personal_status'
    if re.search(r'\b(офис|отделени|банкомат|адрес|рядом|город|москва|санкт|работает ли|график)\b', ql):
        return 'branch_location'
    if TARIFF_HINT_RE.search(ql):
        return 'tariff'
    if LEGAL_HINT_RE.search(ql):
        return 'legal'
    if re.search(r'\b(как|где|куда|что делать|почему|не могу|не работает|ошибка|подключить|отключить|изменить|сменить|восстановить|закрыть|открыть|оформить|пополнить|перевести|получить)\b', ql):
        return 'procedure'
    if re.search(r'\b(карта|кредит|ипотек|вклад|счет|счёт|инвест|страхов|альфа|пакет|подписк|продукт)\b', ql):
        return 'product'
    return 'fact'


def expand_synonyms(q: str) -> str:
    ql = normalize_terms(q.lower())
    additions = []
    pairs = [
        ('кэшбэк', 'кешбэк cashback'), ('кешбэк', 'кэшбэк cashback'),
        ('sms', 'смс сообщение уведомление'), ('push', 'пуш уведомление'),
        ('pin', 'пин код pin-код'), ('альфа-онлайн', 'альфа онлайн интернет банк приложение'),
        ('счет', 'счёт реквизиты'), ('счёт', 'счет реквизиты'),
        ('справка', 'документ выписка'), ('лимит', 'ограничение'),
    ]
    for key, add in pairs:
        if key in ql:
            additions.append(add)
    return (q + ' ' + ' '.join(additions)).strip()


def build_query_variants(q: str, intent: str) -> List[str]:
    qn = normalize_query(q)
    variants = [qn]
    exp = expand_synonyms(qn)
    if exp != qn:
        variants.append(exp)
    if intent == 'procedure':
        variants.append(qn + ' инструкция шаги как сделать в приложении альфа-онлайн')
    elif intent == 'tariff':
        variants.append(qn + ' тариф комиссия лимит стоимость условия')
    elif intent == 'legal':
        variants.append(qn + ' условия правила договор пункт')
    elif intent == 'personal_status':
        variants.append(qn + ' общие правила условия как проверить статус')
    elif intent == 'branch_location':
        variants.append(qn + ' офис отделение банкомат адрес график')
    elif intent == 'product':
        variants.append(qn + ' условия продукта оформление требования')
    # dedupe preserving order
    out = []
    seen = set()
    for v in variants:
        key = v.lower()
        if key not in seen:
            out.append(v)
            seen.add(key)
    return out[:3]


def query_terms(q: str) -> set:
    return set(tokenize(q, lemmatize=True))


def lexical_overlap(query: str, text: str) -> float:
    qt = query_terms(query)
    if not qt:
        return 0.0
    tt = set(tokenize(text, lemmatize=True))
    return len(qt & tt) / max(len(qt), 1)

print(classify_intent('Как отключить смс уведомления?'), build_query_variants('Как отключить смс уведомления?', 'procedure'))

procedure ['Как отключить sms уведомления?', 'Как отключить sms уведомления? смс сообщение уведомление', 'Как отключить sms уведомления? инструкция шаги как сделать в приложении альфа-онлайн']


In [13]:
# === CELL 10 — hybrid retrieval with RRF ===
SOURCE_WEIGHTS = {
    'support_faq': 1.18,
    'support_help': 1.15,
    'product_page': 1.05,
    'general_page': 1.00,
    'legal_tariff': 0.96,
    'legal': 0.90,
    'blog_article': 0.78,
    'news_article': 0.76,
}

INTENT_SOURCE_BOOST = {
    'procedure': {'support_faq': 1.18, 'support_help': 1.16, 'product_page': 1.02},
    'tariff': {'legal_tariff': 1.22, 'support_faq': 1.08, 'product_page': 1.08},
    'legal': {'legal': 1.18, 'legal_tariff': 1.15, 'support_faq': 1.02},
    'product': {'product_page': 1.16, 'support_faq': 1.08},
    'personal_status': {'support_faq': 1.15, 'support_help': 1.10},
    'branch_location': {'support_faq': 1.08, 'general_page': 1.05},
}


def source_boost(stype: str, intent: str) -> float:
    base = SOURCE_WEIGHTS.get(stype, 1.0)
    return base * INTENT_SOURCE_BOOST.get(intent, {}).get(stype, 1.0)


def bm25_search(q: str, top_n: int) -> List[Tuple[int, float]]:
    toks = tokenize(q, RETRIEVAL_CONFIG['use_lemmatization'])
    scores = bm25.get_scores(toks)
    if len(scores) == 0:
        return []
    idx = np.argpartition(scores, -min(top_n, len(scores)))[-min(top_n, len(scores)):]
    idx = idx[np.argsort(scores[idx])[::-1]]
    return [(int(i), float(scores[i])) for i in idx if scores[i] > 0]


def semantic_search(q: str, top_n: int) -> List[Tuple[int, float]]:
    q_emb = embedder.encode([q], is_query=True, batch_size=1)
    scores, ids = index.search(q_emb.astype('float32'), min(top_n, len(chunks)))
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0]) if i >= 0]


def rrf_merge(rankings: List[List[Tuple[int, float]]], intent: str, rrf_k: int = 60) -> List[Dict[str, Any]]:
    acc = defaultdict(float)
    raw = defaultdict(dict)
    for stream_id, ranking in enumerate(rankings):
        for rank, (cid, score) in enumerate(ranking, start=1):
            boost = source_boost(chunks[cid].source_type, intent)
            acc[cid] += boost / (rrf_k + rank)
            raw[cid][f'stream_{stream_id}'] = score
    out = []
    for cid, score in acc.items():
        out.append({'chunk_id': cid, 'rrf_score': float(score), 'raw_scores': raw[cid]})
    out.sort(key=lambda x: x['rrf_score'], reverse=True)
    return out


def retrieve_candidates(query: str, intent: str) -> List[Dict[str, Any]]:
    variants = build_query_variants(query, intent)
    rankings = []
    for qv in variants:
        rankings.append(semantic_search(qv, RETRIEVAL_CONFIG['semantic_top_n']))
        rankings.append(bm25_search(qv, RETRIEVAL_CONFIG['bm25_top_n']))
    merged = rrf_merge(rankings, intent, RETRIEVAL_CONFIG['rrf_k'])
    # lexical tie-break / filter boost
    for item in merged:
        c = chunks[item['chunk_id']]
        item['lex_overlap'] = lexical_overlap(query, c.retrieval_text)
        item['hybrid_score'] = item['rrf_score'] * (1.0 + 0.12 * item['lex_overlap'])
    merged.sort(key=lambda x: x['hybrid_score'], reverse=True)
    return merged[:RETRIEVAL_CONFIG['final_retrieval_top_n']]

# Smoke test
_test_q = questions_run.iloc[0]['query']
_test_intent = classify_intent(_test_q)
_test_cands = retrieve_candidates(_test_q, _test_intent)[:3]
print('Query:', _test_q, 'intent:', _test_intent)
for r in _test_cands:
    c = chunks[r['chunk_id']]
    print(r['hybrid_score'], c.web_id, c.source_type, c.title[:80], '=>', c.text[:160])

Query: Номер счета intent: fact
0.07860914695340501 1721 support_faq Платежи в рублях => озванной лицензией в реквизите 17 «Номер счета получателя средств» в платеже на бумажном носителе не заполняется/заполняется согласно настройкам интернет-банкин
0.07725879430253214 372 support_faq Номер расчетного счета: что это такое и расшифровка - Блог 🅰️ Альфа-Банка => как платежные системы банка проверяют на соответствие ИНН получателя, БИК, номер корреспондентского и расчетного счета. При любых несоответствиях перевод будет 
0.0752537302093521 372 support_faq Номер расчетного счета: что это такое и расшифровка - Блог 🅰️ Альфа-Банка => программе. Если понимать, что означают цифры в расчетном счете, это снизит риск того, что данные будут указаны неверно. Изучив платежные реквизиты партнера, мож


In [14]:
# === CELL 11 — Qwen3 reranker ===
class QwenReranker:
    def __init__(self, model_name: str, use_4bit: bool = False, max_length: int = 4096):
        self.model_name = model_name
        self.max_length = max_length
        print('Loading reranker:', model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, padding_side='left')
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        quant_config = None
        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        if use_4bit and torch.cuda.is_available():
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type='nf4',
            )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            device_map='auto' if torch.cuda.is_available() else None,
            quantization_config=quant_config,
            trust_remote_code=True,
        )
        self.model.eval()
        self.yes_id = self.tokenizer.encode('yes', add_special_tokens=False)[-1]
        self.no_id = self.tokenizer.encode('no', add_special_tokens=False)[-1]
        self.system = 'Judge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".'
        self.task = 'Given a Russian banking support question, decide whether the document contains facts needed to answer it.'

    def _format_pair(self, query: str, doc: str) -> str:
        user = f'<Instruct>: {self.task}\n<Query>: {query}\n<Document>: {doc}'
        messages = [{'role': 'system', 'content': self.system}, {'role': 'user', 'content': user}]
        try:
            return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    @torch.no_grad()
    def score(self, query: str, docs: List[str], batch_size: int = 4) -> List[float]:
        scores = []
        for i in range(0, len(docs), batch_size):
            texts = [self._format_pair(query, d) for d in docs[i:i+batch_size]]
            inputs = self.tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt',
            ).to(self.model.device)
            logits = self.model(**inputs).logits[:, -1, :]
            raw = logits[:, self.yes_id] - logits[:, self.no_id]
            scores.extend(raw.detach().float().cpu().tolist())
        return scores


def sigmoid(x: float) -> float:
    try:
        return 1 / (1 + math.exp(-float(x)))
    except OverflowError:
        return 0.0 if x < 0 else 1.0

reranker = QwenReranker(
    RERANKER_MODEL_NAME,
    use_4bit=RERANKER_CONFIG['use_4bit'],
    max_length=RERANKER_CONFIG['max_length'],
)


def rerank_candidates(query: str, candidates: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    top = candidates[:RERANKER_CONFIG['input_top_n']]
    docs = []
    for item in top:
        c = chunks[item['chunk_id']]
        # Reranker получает компактный, но богатый doc.
        docs.append(f'Заголовок: {c.title}\nРаздел: {c.section_title}\nТип: {c.source_type}\nТекст: {c.text}')
    raw_scores = reranker.score(query, docs, batch_size=RERANKER_CONFIG['batch_size']) if docs else []
    out = []
    for item, raw in zip(top, raw_scores):
        obj = dict(item)
        obj['rerank_raw'] = float(raw)
        obj['rerank_sigmoid'] = sigmoid(raw)
        # Небольшая смесь с hybrid_score для стабильности на шумных logits.
        obj['final_score'] = 0.88 * obj['rerank_sigmoid'] + 0.12 * min(1.0, obj['hybrid_score'] * 100)
        out.append(obj)
    out.sort(key=lambda x: x['final_score'], reverse=True)
    return out[:RERANKER_CONFIG['rerank_top_k']]

# Smoke test reranker on 3 docs
_test_rr = rerank_candidates(_test_q, _test_cands)
for r in _test_rr:
    c = chunks[r['chunk_id']]
    print(round(r['rerank_sigmoid'], 3), c.title[:70], c.text[:180])

Loading reranker: Qwen/Qwen3-Reranker-0.6B


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

0.004 Платежи в рублях озванной лицензией в реквизите 17 «Номер счета получателя средств» в платеже на бумажном носителе не заполняется/заполняется согласно настройкам интернет-банкинга в реквизите 24 «Н
0.004 Номер расчетного счета: что это такое и расшифровка - Блог 🅰️ Альфа-Ба как платежные системы банка проверяют на соответствие ИНН получателя, БИК, номер корреспондентского и расчетного счета. При любых несоответствиях перевод будет заблокирован. В этом
0.004 Номер расчетного счета: что это такое и расшифровка - Блог 🅰️ Альфа-Ба программе. Если понимать, что означают цифры в расчетном счете, это снизит риск того, что данные будут указаны неверно. Изучив платежные реквизиты партнера, можно выяснить как мини


In [15]:
# === CELL 12 — evidence selection and compression ===
chunk_by_id = {c.chunk_id: c for c in chunks}
chunks_by_web = defaultdict(list)
for c in chunks:
    chunks_by_web[c.web_id].append(c)
for web_id in chunks_by_web:
    chunks_by_web[web_id].sort(key=lambda x: x.chunk_index)

chunk_pos_by_web = {(c.web_id, c.chunk_index): c.chunk_id for c in chunks}


def diversify_top_chunks(reranked: List[Dict[str, Any]], final_top_k: int, max_per_web_id: int) -> List[Dict[str, Any]]:
    selected = []
    counts = Counter()
    for item in reranked:
        c = chunks[item['chunk_id']]
        if counts[c.web_id] >= max_per_web_id:
            continue
        selected.append(item)
        counts[c.web_id] += 1
        if len(selected) >= final_top_k:
            break
    if len(selected) < final_top_k:
        selected_ids = {x['chunk_id'] for x in selected}
        for item in reranked:
            if item['chunk_id'] not in selected_ids:
                selected.append(item)
                if len(selected) >= final_top_k:
                    break
    return selected


def expand_neighbors(selected: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    seen = set()
    for item in selected:
        c = chunks[item['chunk_id']]
        for delta in range(-EVIDENCE_CONFIG['neighbor_window'], EVIDENCE_CONFIG['neighbor_window'] + 1):
            nid = chunk_pos_by_web.get((c.web_id, c.chunk_index + delta))
            if nid is None or nid in seen:
                continue
            base = dict(item)
            base['chunk_id'] = nid
            base['is_neighbor'] = (delta != 0)
            if delta != 0:
                base['final_score'] = base.get('final_score', 0.0) * 0.82
                base['rerank_sigmoid'] = base.get('rerank_sigmoid', 0.0) * 0.82
            out.append(base)
            seen.add(nid)
            if len(out) >= EVIDENCE_CONFIG['max_expanded_chunks']:
                return out
    return out


def important_sentence_score(query: str, sent: str, chunk_score: float, intent: str) -> float:
    score = 0.55 * lexical_overlap(query, sent) + 0.45 * chunk_score
    low = sent.lower()
    if intent in {'tariff', 'legal'} and (TARIFF_HINT_RE.search(low) or LEGAL_HINT_RE.search(low)):
        score += 0.12
    if intent == 'procedure' and re.search(r'\b(зайдите|откройте|нажмите|выберите|перейдите|укажите|подтвердите|можно|нужно)\b', low):
        score += 0.10
    if re.search(r'\d|%|₽|руб|дн|час|месяц|год', low):
        score += 0.05
    return score


def context_budget(intent: str) -> int:
    if intent == 'legal':
        return EVIDENCE_CONFIG['max_context_chars_legal']
    if intent == 'tariff':
        return EVIDENCE_CONFIG['max_context_chars_tariff']
    if intent == 'procedure':
        return EVIDENCE_CONFIG['max_context_chars_procedure']
    return EVIDENCE_CONFIG['max_context_chars_default']


def compress_evidence(query: str, reranked: List[Dict[str, Any]], intent: str) -> Tuple[str, List[Dict[str, Any]]]:
    selected = diversify_top_chunks(reranked, EVIDENCE_CONFIG['final_top_k'], EVIDENCE_CONFIG['max_chunks_per_web_id'])
    expanded = expand_neighbors(selected)
    sent_items = []
    for item in expanded:
        c = chunks[item['chunk_id']]
        sents = split_sentences(c.text)
        for sent in sents:
            sent = sent.strip()
            if len(sent) < 20:
                continue
            s = important_sentence_score(query, sent, float(item.get('rerank_sigmoid', 0.0)), intent)
            sent_items.append({
                'score': s,
                'sent': sent,
                'chunk_id': c.chunk_id,
                'web_id': c.web_id,
                'title': c.title,
                'section_title': c.section_title,
                'source_type': c.source_type,
                'url': c.url,
                'chunk_score': item.get('rerank_sigmoid', 0.0),
            })
    sent_items.sort(key=lambda x: x['score'], reverse=True)
    # Dedupe near-identical sentences.
    chosen = []
    seen = set()
    total = 0
    budget = context_budget(intent)
    for it in sent_items:
        norm = re.sub(r'\W+', '', it['sent'].lower())[:180]
        if norm in seen:
            continue
        block = f"[{len(chosen)+1}] {it['title']} / {it['section_title']}: {it['sent']}"
        if total + len(block) > budget and len(chosen) >= 6:
            continue
        chosen.append(it)
        seen.add(norm)
        total += len(block)
        if len(chosen) >= EVIDENCE_CONFIG['max_evidence_sentences'] or total >= budget:
            break
    # Восстанавливаем более естественный порядок: по web_id/chunk_id, но сохраняя только выбранные.
    chosen.sort(key=lambda x: (x['web_id'], x['chunk_id']))
    lines = []
    for i, it in enumerate(chosen, start=1):
        lines.append(f"[{i}] Источник: {it['title']} | Раздел: {it['section_title']} | Тип: {it['source_type']}\nФакт: {it['sent']}")
    context = '\n\n'.join(lines).strip()
    return context[:budget], chosen


def answerability(query: str, reranked: List[Dict[str, Any]], context: str, intent: str) -> Tuple[bool, Dict[str, Any]]:
    top_sigmoid = float(reranked[0]['rerank_sigmoid']) if reranked else 0.0
    overlap = lexical_overlap(query, context)
    ctx_len = len(context)
    has_numbers_or_terms = bool(re.search(r'\d|%|₽|руб|комисс|лимит|срок|услов|можно|нужно|зайдите|откройте', context.lower()))
    no_support = (
        ctx_len < ANSWERABILITY_CONFIG['min_context_chars'] or
        (top_sigmoid < ANSWERABILITY_CONFIG['very_low_top_sigmoid'] and overlap < ANSWERABILITY_CONFIG['min_query_overlap']) or
        (top_sigmoid < ANSWERABILITY_CONFIG['min_top_sigmoid'] and overlap < ANSWERABILITY_CONFIG['min_query_overlap'] and not has_numbers_or_terms)
    )
    # Для personal_status не пишем no-answer, если нашли общие правила.
    if intent == 'personal_status' and ctx_len >= 200 and overlap >= 0.02:
        no_support = False
    return (not no_support), {
        'top_sigmoid': top_sigmoid,
        'query_context_overlap': overlap,
        'context_chars': ctx_len,
        'has_numbers_or_terms': has_numbers_or_terms,
    }

# Smoke test evidence
_test_reranked = rerank_candidates(_test_q, retrieve_candidates(_test_q, _test_intent))
_test_ctx, _test_ev = compress_evidence(_test_q, _test_reranked, _test_intent)
print('answerable:', answerability(_test_q, _test_reranked, _test_ctx, _test_intent))
print(_test_ctx[:1000])

answerable: (True, {'top_sigmoid': 0.004070137715896128, 'query_context_overlap': 1.0, 'context_chars': 3800, 'has_numbers_or_terms': True})
[1] Источник: Номер расчетного счета: что это такое и расшифровка - Блог 🅰️ Альфа-Банка | Раздел: Номер расчетного счета: что это такое и расшифровка - Блог 🅰️ Альфа-Банка | Тип: support_faq
Факт: Потребуется уточнить БИК и по нему найти название банка 0 001 782 - внутренний номер владельца расчетного счета в банке Хотя сам по себе номер счета и не дает детальной информации о бизнесе, его расшифровка поможет получить общее представление о контрагенте перед началом сотрудничества.

[2] Источник: Номер расчетного счета: что это такое и расшифровка - Блог 🅰️ Альфа-Банка | Раздел: Номер расчетного счета: что это такое и расшифровка - Блог 🅰️ Альфа-Банка | Тип: support_faq
Факт: Есть несколько вариантов выяснить реквизиты банковского счета организации или ИП.

[3] Источник: 💳 IBAN: что это такое и как расшифровать банковский реквизит счета? Рассказывае

In [16]:
# === CELL 13 — generator-first Qwen answerer ===
class QwenGenerator:
    def __init__(self, model_name: str):
        self.model_name = model_name
        print('Loading generator:', model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        quant_config = None
        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        if GENERATION_CONFIG['load_in_4bit'] and torch.cuda.is_available():
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type='nf4',
            )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            device_map='auto' if torch.cuda.is_available() else None,
            quantization_config=quant_config,
            trust_remote_code=True,
        )
        self.model.eval()

    def build_messages(self, query: str, context: str, intent: str, target_chars: int) -> List[Dict[str, str]]:
        system = (
            'Ты отвечаешь по базе знаний Альфа-Банка. Используй только факты из блока КОНТЕКСТ. '
            'Не выдумывай условия, сроки, комиссии, статусы, названия продуктов и цифры. '
            'Если в контексте есть важные ограничения, исключения, шаги, сроки, комиссии или лимиты — включи их в ответ. '
            'Если вопрос про личный статус пользователя, а в контексте есть только общие правила, дай общий ответ и явно скажи, что персональные данные тебе недоступны. '
            'Пиши по-русски, естественно, без канцелярита. '
            f'Если информации в контексте действительно недостаточно, ответь ровно: {NO_ANSWER}'
        )
        user = f'''ВОПРОС:
{query}

ТИП ВОПРОСА: {intent}

КОНТЕКСТ:
{context}

ОГРАНИЧЕНИЯ:
- не более {target_chars} символов;
- не пропускай существенные условия, сроки, комиссии, лимиты, исключения и шаги;
- не цитируй длинные фрагменты дословно;
- не добавляй информацию вне контекста;
- не упоминай номера источников и слово «контекст».

СФОРМИРУЙ ФИНАЛЬНЫЙ ОТВЕТ:'''
        return [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]

    @torch.no_grad()
    def generate(self, query: str, context: str, intent: str, target_chars: int) -> str:
        messages = self.build_messages(query, context, intent, target_chars)
        try:
            prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True, max_length=12000).to(self.model.device)
        out = self.model.generate(
            **inputs,
            max_new_tokens=GENERATION_CONFIG['max_new_tokens'],
            temperature=GENERATION_CONFIG['temperature'],
            top_p=GENERATION_CONFIG['top_p'],
            repetition_penalty=GENERATION_CONFIG['repetition_penalty'],
            do_sample=GENERATION_CONFIG['do_sample'],
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        gen = out[0, inputs['input_ids'].shape[1]:]
        text = self.tokenizer.decode(gen, skip_special_tokens=True)
        return clean_generated_answer(text, target_chars)


def answer_budget(intent: str) -> int:
    if intent == 'legal':
        return GENERATION_CONFIG['answer_cap_legal']
    if intent == 'tariff':
        return GENERATION_CONFIG['answer_cap_tariff']
    if intent == 'procedure':
        return GENERATION_CONFIG['answer_cap_procedure']
    if intent == 'fact':
        return GENERATION_CONFIG['answer_cap_fact']
    return GENERATION_CONFIG['answer_cap_default']


def clean_generated_answer(text: str, max_chars: int) -> str:
    t = str(text).strip()
    t = re.sub(r'^(ответ|финальный ответ)\s*[:：]\s*', '', t, flags=re.I).strip()
    t = re.sub(r'<[^>]+>', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip(' "\'')
    # Убираем markdown bullets, если ответ короткий; списки оставляем, но переводим в компактный вид.
    t = re.sub(r'\s*[-•]\s+', '; ', t).strip('; ')
    if not t:
        return NO_ANSWER
    if NO_ANSWER.lower() in t.lower() and len(t) < 80:
        return NO_ANSWER
    if len(t) <= max_chars:
        return t
    cut = t[:max_chars]
    # Обрезаем по последнему предложению/разделителю.
    last = max(cut.rfind('.'), cut.rfind('!'), cut.rfind('?'), cut.rfind(';'))
    if last > max_chars * 0.55:
        cut = cut[:last+1]
    return cut.strip() or t[:max_chars].strip()


def extract_numbers(text: str) -> set:
    return set(re.findall(r'\d+[\d\s.,]*\s*(?:%|₽|руб(?:лей|ля|ль)?|дн(?:ей|я)?|час(?:ов|а)?|мес(?:яц(?:ев|а)?)?|год(?:а|ов)?|мин(?:ут)?)?', str(text).lower()))


def groundedness_ok(answer: str, context: str) -> Tuple[bool, Dict[str, Any]]:
    if answer == NO_ANSWER:
        return True, {'new_numbers': [], 'answer_context_overlap': 1.0}
    ans_nums = {x.strip() for x in extract_numbers(answer) if x.strip()}
    ctx_nums = {x.strip() for x in extract_numbers(context) if x.strip()}
    # Разрешаем голые числа, если число есть как подстрока в контексте.
    new_nums = []
    for n in ans_nums:
        n_plain = re.sub(r'\s+', ' ', n).strip()
        if n_plain and n_plain not in ctx_nums and n_plain not in context.lower():
            new_nums.append(n_plain)
    ov = lexical_overlap(answer, context)
    ok = (len(new_nums) == 0) and (ov >= 0.08 or len(answer) < 120)
    return ok, {'new_numbers': new_nums, 'answer_context_overlap': ov}

generator = QwenGenerator(GENERATOR_MODEL_NAME)

Loading generator: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [17]:
# === CELL 14 — end-to-end answer function ===
def generate_safe_general_answer(query: str, context: str, intent: str) -> str:
    # Аварийный режим: если генерация/верификация сломалась, не возвращаем сырой extractive chunk,
    # но даем короткий grounded ответ из самых релевантных фактов.
    if not context or len(context) < 120:
        return NO_ANSWER
    sents = []
    for line in context.split('\n'):
        if line.startswith('Факт:'):
            sents.append(line.replace('Факт:', '').strip())
    if not sents:
        return NO_ANSWER
    answer = ' '.join(sents[:4])
    return clean_generated_answer(answer, answer_budget(intent))


def answer_one(query: str, q_id: Optional[int] = None, verbose: bool = False) -> Tuple[str, Dict[str, Any]]:
    t0 = time.time()
    q = normalize_query(query)
    intent = classify_intent(q)
    diag: Dict[str, Any] = {'q_id': q_id, 'query': query, 'intent': intent}
    try:
        candidates = retrieve_candidates(q, intent)
        diag['n_candidates'] = len(candidates)
        reranked = rerank_candidates(q, candidates)
        diag['n_reranked'] = len(reranked)
        context, evidence = compress_evidence(q, reranked, intent)
        can_answer, ans_diag = answerability(q, reranked, context, intent)
        diag.update(ans_diag)
        diag['n_evidence_sentences'] = len(evidence)
        if verbose:
            print('INTENT:', intent, 'ANSWERABLE:', can_answer, ans_diag)
            print(context[:1500])
        if not can_answer:
            answer = NO_ANSWER
            diag['mode'] = 'no_answer_by_answerability'
        else:
            target_chars = answer_budget(intent)
            answer = generator.generate(q, context, intent, target_chars)
            ok, gdiag = groundedness_ok(answer, context)
            diag.update(gdiag)
            if not ok and answer != NO_ANSWER:
                # Один repair pass: просим переписать без неподтвержденных чисел/фактов.
                repair_context = context + "\n\nВажно: в предыдущем черновике были неподтвержденные числа или факты. Перепиши ответ строго по фактам выше."
                repaired = generator.generate(q, repair_context, intent, target_chars)
                ok2, gdiag2 = groundedness_ok(repaired, context)
                diag['repair_used'] = True
                diag['repair_ok'] = ok2
                diag['repair_new_numbers'] = gdiag2.get('new_numbers', [])
                if ok2:
                    answer = repaired
                else:
                    # Последний grounded fallback, но не как основной режим.
                    answer = generate_safe_general_answer(q, context, intent)
                    diag['mode'] = 'safe_general_after_failed_grounding'
            if 'mode' not in diag:
                diag['mode'] = 'generator_first'
        answer = clean_generated_answer(answer, answer_budget(intent))
        diag['answer_chars'] = len(answer)
        diag['answer'] = answer
        diag['elapsed_sec'] = round(time.time() - t0, 3)
        return answer, diag
    except RuntimeError as e:
        # CUDA OOM: аккуратная очистка и no-answer, чтобы прогон не умер.
        if 'out of memory' in str(e).lower():
            print('CUDA OOM on q_id', q_id, '- clearing cache')
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            diag['error'] = 'CUDA OOM'
            diag['mode'] = 'error_no_answer'
            diag['answer'] = NO_ANSWER
            return NO_ANSWER, diag
        raise
    except Exception as e:
        diag['error'] = repr(e)
        diag['mode'] = 'error_no_answer'
        diag['answer'] = NO_ANSWER
        return NO_ANSWER, diag

# Smoke test full answer on one query
smoke_q = questions_run.iloc[0]['query']
smoke_answer, smoke_diag = answer_one(smoke_q, int(questions_run.iloc[0]['q_id']), verbose=True)
print('\nANSWER:', smoke_answer)
print('DIAG:', smoke_diag)

INTENT: fact ANSWERABLE: True {'top_sigmoid': 0.004070137715896128, 'query_context_overlap': 1.0, 'context_chars': 3800, 'has_numbers_or_terms': True}
[1] Источник: Номер расчетного счета: что это такое и расшифровка - Блог 🅰️ Альфа-Банка | Раздел: Номер расчетного счета: что это такое и расшифровка - Блог 🅰️ Альфа-Банка | Тип: support_faq
Факт: Потребуется уточнить БИК и по нему найти название банка 0 001 782 - внутренний номер владельца расчетного счета в банке Хотя сам по себе номер счета и не дает детальной информации о бизнесе, его расшифровка поможет получить общее представление о контрагенте перед началом сотрудничества.

[2] Источник: Номер расчетного счета: что это такое и расшифровка - Блог 🅰️ Альфа-Банка | Раздел: Номер расчетного счета: что это такое и расшифровка - Блог 🅰️ Альфа-Банка | Тип: support_faq
Факт: Есть несколько вариантов выяснить реквизиты банковского счета организации или ИП.

[3] Источник: 💳 IBAN: что это такое и как расшифровать банковский реквизит счета? Р

In [18]:
# === CELL 15 — generation loop with checkpointing ===
def load_checkpoint(path: Path) -> Tuple[Dict[int, str], List[Dict[str, Any]]]:
    if not path.exists() or FORCE_REGENERATE:
        return {}, []
    df = pd.read_csv(path)
    answers = dict(zip(df['q_id'].astype(int), df['answer_new'].astype(str)))
    diag_cols = [c for c in df.columns if c not in {'answer_new'}]
    diags = df[diag_cols].to_dict('records') if diag_cols else []
    print('Loaded checkpoint:', len(answers))
    return answers, diags


def save_checkpoint(path: Path, answers: Dict[int, str], diagnostics: List[Dict[str, Any]]):
    rows = []
    diag_by_qid = {int(d.get('q_id')): d for d in diagnostics if d.get('q_id') is not None}
    for qid, ans in answers.items():
        row = {'q_id': int(qid), 'answer_new': ans}
        if qid in diag_by_qid:
            for k, v in diag_by_qid[qid].items():
                if k not in row and k != 'answer':
                    row[k] = json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else v
        rows.append(row)
    pd.DataFrame(rows).sort_values('q_id').to_csv(path, index=False)

answers, diagnostics = load_checkpoint(CHECKPOINT_PATH)

rows = list(questions_run[['q_id', 'query']].itertuples(index=False, name=None))
for idx, (qid, query) in enumerate(tqdm(rows, desc='Generating answers')):
    qid = int(qid)
    if qid in answers and not FORCE_REGENERATE:
        continue
    ans, diag = answer_one(query, qid, verbose=False)
    answers[qid] = ans
    diagnostics.append(diag)
    if (idx + 1) % GENERATION_CONFIG['save_every'] == 0:
        save_checkpoint(CHECKPOINT_PATH, answers, diagnostics)
        print('Checkpoint saved:', len(answers))

save_checkpoint(CHECKPOINT_PATH, answers, diagnostics)
print('Done. Answers:', len(answers), 'checkpoint:', CHECKPOINT_PATH)

Generating answers:   0%|          | 0/6977 [00:00<?, ?it/s]

Checkpoint saved: 25
Checkpoint saved: 50
Checkpoint saved: 75
Checkpoint saved: 100
Checkpoint saved: 125
Checkpoint saved: 150
Checkpoint saved: 175
Checkpoint saved: 200
Checkpoint saved: 225
Checkpoint saved: 250
Checkpoint saved: 275
Checkpoint saved: 300
Checkpoint saved: 325
Checkpoint saved: 350
Checkpoint saved: 375
Checkpoint saved: 400
Checkpoint saved: 425
Checkpoint saved: 450
Checkpoint saved: 475
Checkpoint saved: 500
Checkpoint saved: 525
Checkpoint saved: 550
Checkpoint saved: 575
Checkpoint saved: 600
Checkpoint saved: 625
Checkpoint saved: 650
Checkpoint saved: 675
Checkpoint saved: 700
Checkpoint saved: 725
Checkpoint saved: 750
Checkpoint saved: 775
Checkpoint saved: 800
Checkpoint saved: 825
Checkpoint saved: 850
Checkpoint saved: 875
Checkpoint saved: 900
Checkpoint saved: 925
Checkpoint saved: 950
Checkpoint saved: 975
Checkpoint saved: 1000
Checkpoint saved: 1025
Checkpoint saved: 1050
Checkpoint saved: 1075
Checkpoint saved: 1100
Checkpoint saved: 1125
Checkpo

In [19]:
# === CELL 16 — build and validate submission ===
submission = questions_run[['q_id']].copy()
submission['answer_new'] = submission['q_id'].map(answers).fillna(NO_ANSWER).astype(str)

# Если запускали DEV_N, сохраняем dev сабмит отдельно, чтобы случайно не отправить неполный файл.
out_path = SUBMISSION_PATH if DEV_N is None else DATA_DIR / f'submission_{RUN_NAME}_DEV{DEV_N}.csv'
diag_path = DIAGNOSTICS_PATH if DEV_N is None else DATA_DIR / f'diagnostics_{RUN_NAME}_DEV{DEV_N}.csv'

# Формальная валидация.
assert list(submission.columns) == ['q_id', 'answer_new']
assert submission['q_id'].is_unique
assert submission['answer_new'].isna().sum() == 0
if DEV_N is None:
    assert submission.shape[0] == questions.shape[0], f'Expected {questions.shape[0]}, got {submission.shape[0]}'

submission.to_csv(out_path, index=False)

diag_df = pd.DataFrame(diagnostics)
if not diag_df.empty:
    diag_df.to_csv(diag_path, index=False)

print('Saved submission:', out_path)
print('Saved diagnostics:', diag_path)
print('shape:', submission.shape)
print('No-answer count:', int((submission['answer_new'] == NO_ANSWER).sum()))
print('Unique answers:', submission['answer_new'].nunique())
print('Answer length stats:')
display(submission['answer_new'].str.len().describe(percentiles=[.25, .5, .75, .9, .95, .99]))
display(submission.head(10))

Saved submission: /content/drive/MyDrive/RAG/submission_qwen_generator_first_v1.csv
Saved diagnostics: /content/drive/MyDrive/RAG/diagnostics_qwen_generator_first_v1.csv
shape: (6977, 2)
No-answer count: 2710
Unique answers: 4191
Answer length stats:


count    6977.000000
mean      164.093737
std       168.538276
min        11.000000
25%        11.000000
50%       141.000000
75%       263.000000
90%       359.000000
95%       442.000000
99%       771.480000
max      1050.000000
Name: answer_new, dtype: float64

,q_id,answer_new
0,1,Номер счета — это уникальный код банковского с...
1,2,"Чтобы узнать бик и счет организации, можно обр..."
2,3,Секретный код не приходит в SMS или push уведо...
3,4,"Нет ответа. В контексте нет информации о том, ..."
4,5,Вы сможете пользоваться кредитной картой после...
5,6,Нет ответа.
6,7,"Закрыть бизнес счет можно в Альфа-Банке, но ин..."
7,8,Нет ответа. Зарплатный счет относится к расчет...
8,9,Дата начала пользования кредитной картой начин...
9,10,Нет ответа.


In [ ]:
# === CELL 17 — lightweight diagnostics / examples ===
if not diag_df.empty:
    print('Modes:')
    display(diag_df['mode'].value_counts(dropna=False).head(20))
    print('Intents:')
    display(diag_df['intent'].value_counts(dropna=False))
    show_cols = ['q_id', 'intent', 'mode', 'top_sigmoid', 'query_context_overlap', 'context_chars', 'answer_chars', 'answer']
    display(diag_df[[c for c in show_cols if c in diag_df.columns]].head(20))

# Посмотреть самые короткие non-empty ответы и потенциальные повторы.
non_empty = submission.copy()
non_empty['len'] = non_empty['answer_new'].str.len()
print('Shortest answers:')
display(non_empty.sort_values('len').head(20))
print('Most common answers:')
display(submission['answer_new'].value_counts().head(20))

## Рекомендуемые первые прогоны

1. `DEV_N = 50` — проверить, что модели грузятся и сабмит сохраняется.
2. `DEV_N = 300` — посмотреть no-answer rate, повторы, среднюю длину, качество evidence.
3. Полный прогон с дефолтом: `Qwen3-Embedding-0.6B + Qwen3-Reranker-0.6B + Qwen2.5-7B-Instruct`.
4. Если есть A100 или достаточно времени: заменить `EMBEDDING_MODEL_NAME` на `Qwen/Qwen3-Embedding-4B`.
5. Если генератор слишком медленный, оставьте retrieval/rerank как есть, но уменьшите `GENERATION_CONFIG['max_new_tokens']` до 180 и answer caps до 750–850.